# Healthcare company website summarizer

A Week 1 exercise that summarizes a company’s goals and examples from its website, with light humor about corporate jargon.

Use the course Python environment and keep `OPENAI_API_KEY` in the repository-root `.env` file. Run the cells in order. Change the URL or question to try another company. This reads the first 2,000 characters of one page, so it may not contain enough information to answer the question.


In [ ]:
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(override=True)
openai = OpenAI()


In [ ]:
def fetch_website_contents(url):
    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=30,
    )
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.get_text(strip=True) if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


In [ ]:
# Step 1: Create your prompts

system_prompt = """You are a healthcare information assistant.
Summarize the provided website text, focusing on the user's question.
Use a professional tone and plain language. Explain medical terms clearly.
Add light, witty humor about corporate jargon or website buzzwords, while keeping the facts accurate. Avoid jokes about patients, illnesses, or treatments.
Use only information supported by the provided text, and say when it does not answer the user's question.
Respond in markdown without wrapping it in a code block."""
user_prompt = """
    What are the main goals of the company, and what examples show how the company has put those goals into practice?
    
"""


In [ ]:
# Step 2: Fetch the website and build the messages
website = fetch_website_contents("https://www.optum.com/en/")
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt + "\n\nWebsite text:\n" + website},
]


In [ ]:
# Step 3: Call OpenAI
response = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
)


In [ ]:
# Step 4: Display the returned answer
summary = response.choices[0].message.content
display(Markdown(summary))
